**Last modified on**: 07/05/2026

**Author**: Onur Serçinoğlu

**Update Logs**:

07/05/2026: Initial version. Replaces the older `4_blast_msa_phylogeny.ipynb`.

**Credits**:

This Jupyter notebook accompanies the Multiple Sequence Alignment lecture prepared for BENG451/BSB511 at Gebze Technical University. Key references:
- Edgar RC (2004). MUSCLE: multiple sequence alignment with high accuracy and high throughput. *Nucleic Acids Res* 32:1792–1797.
- Edgar RC (2022). Muscle5: High-accuracy alignment ensembles enable unbiased assessments of sequence homology and phylogeny. *Nat Commun* 13:6968.
- Madeira F et al. (2024). The EMBL-EBI Job Dispatcher sequence analysis tools framework in 2024. *Nucleic Acids Res* 52:W521–W525.
- Durbin R et al. (1998). *Biological sequence analysis*. Cambridge University Press.

# Multiple Sequence Alignment with MUSCLE

A pairwise alignment (notebooks `3_*`) tells you how two sequences relate. A *multiple* sequence alignment (MSA) tells you how a whole family of related sequences relates — column by column. Once you have an MSA you can:

- Build a profile (`6_2_hmmbuild.ipynb`) — the HMMER side of the course.
- Compute distances and a phylogeny (`5_3_phylogeny.ipynb`) — the next notebook.
- Read off conserved positions that pinpoint catalytic residues, binding sites, or fold-defining contacts.

In this notebook we will:

1. Load a curated set of globin protein sequences.
2. Align them with **MUSCLE 5** running locally as a command-line tool (`subprocess`).
3. Align the *same* sequences with the **EBI MUSCLE web service** (REST API) — useful when you cannot install MUSCLE locally, and the route the term project expects you to use.
4. Compare the two alignments side by side.
5. Extract per-column conservation and gap fraction; identify the conserved heme-binding helices and the flexible loops.
6. Save the canonical alignment for use in `5_3_phylogeny.ipynb`.

For a hands-on refresher on pairwise dynamic programming (the primitive that progressive MSA chains together), see the `alignment-sandbox.html` widget in the course site. For substitution-matrix choice (BLOSUM/PAM), see `scoring-matrices.html`.

In [ ]:
import os
import shutil

# ─── Adjustable paths ────────────────────────────────────────────────────────
WORK_DIR     = "msa_work"
GLOBIN_SET   = "globin_set.fa"                    # offline reference, in exercises/
BLAST_INPUT  = "blast_work/selected_globins.fa"   # produced by 5_1_blast.ipynb (if available)

# Local MUSCLE 5.x binary. Either set MUSCLE_BIN to an explicit path, or leave
# 'muscle' so we pick up whatever is on $PATH (e.g. `conda install -c bioconda muscle`).
MUSCLE_BIN   = shutil.which("muscle") or "muscle"

# EBI politeness — set your contact email here so EBI knows whom to reach if a
# job runs amok. EBI requires this for REST submissions.
EBI_EMAIL    = "your.name@example.org"   # ← REPLACE THIS WITH YOUR EMAIL
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(WORK_DIR, exist_ok=True)

# Decide which input FASTA to align: prefer the BLAST output from 5_1, otherwise
# fall back to the bundled globin_set.fa. Each notebook in the 5_* series uses
# this fallback pattern so it can run standalone.
if os.path.exists(BLAST_INPUT):
    INPUT_FA = BLAST_INPUT
    print(f"Using BLAST output:    {os.path.abspath(INPUT_FA)}")
else:
    INPUT_FA = GLOBIN_SET
    print(f"Falling back to bundled set: {os.path.abspath(INPUT_FA)}")
    print("  (Run 5_1_blast.ipynb first if you want to align your own BLAST hits.)")

print(f"Working directory:     {os.path.abspath(WORK_DIR)}")
print(f"MUSCLE binary:         {MUSCLE_BIN}")

---

In [ ]:
import subprocess
import time
import json
import urllib.request
from urllib.parse import urlencode
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import AlignIO, SeqIO

## 1. Progressive MSA in one paragraph

Suppose you have $N$ sequences and want a single alignment that scores well across all $\binom{N}{2}$ pairwise comparisons. The exact problem is NP-hard. **Progressive alignment** sidesteps that with a heuristic: align the most similar pair first, then progressively merge the rest in the order of a *guide tree*. MUSCLE refines the result with iterative tree re-estimation and column re-alignment.

A standard quality measure for an MSA is the **sum-of-pairs (SP) score**:

$$\text{SP}(A) = \sum_{i<j} \sum_{c} s(A_{i,c}, A_{j,c})$$

summed over every pair of sequences $(i, j)$ and every column $c$, with $s(\cdot, \cdot)$ a substitution score (e.g. BLOSUM62) or a gap penalty. A higher SP score generally means a tighter alignment.

For a refresher on the underlying pairwise dynamic programming, see the `alignment-sandbox.html` widget.

In [ ]:
# Inspect the input FASTA we will align.
records = list(SeqIO.parse(INPUT_FA, "fasta"))
print(f"{len(records)} sequences in {INPUT_FA}\n")
for r in records:
    print(f"  {r.id:30s}  length = {len(r.seq):3d}")

## 2. Aligning with local MUSCLE 5

MUSCLE 5 is invoked as `muscle -align IN.fa -output OUT.aln`. (Note: this is
**not** the MUSCLE 3.x syntax `-in IN.fa -out OUT.aln` — students sometimes
copy-paste old tutorials and get confused. If `muscle -version` reports `3.x`,
upgrade with `conda install -c bioconda muscle=5` or download a binary from
https://github.com/rcedgar/muscle/releases .)

In [ ]:
local_aln = os.path.join(WORK_DIR, "globin_local.aln")

result = subprocess.run(
    [MUSCLE_BIN, "-align", INPUT_FA, "-output", local_aln],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print(f"ERROR: muscle exited with return code {result.returncode}")
    print(result.stderr)
    raise SystemExit("MUSCLE failed — check that the binary is installed and on $PATH.")

print(f"MUSCLE wrote {local_aln} ({os.path.getsize(local_aln)} bytes)")
# MUSCLE 5 prints a progress banner on stderr; we show its tail for context.
print("MUSCLE stderr (tail):")
print("\n".join(result.stderr.splitlines()[-6:]))

In [ ]:
local_alignment = AlignIO.read(local_aln, "fasta")
n_seq = len(local_alignment)
n_col = local_alignment.get_alignment_length()
print(f"Local MUSCLE alignment: {n_seq} sequences × {n_col} columns")
print()
for r in local_alignment:
    pct_gap = 100.0 * str(r.seq).count("-") / n_col
    print(f"  {r.id:30s}  gaps = {pct_gap:5.1f} %")

## 3. Aligning with the EBI MUSCLE web service

The same alignment, run on EBI's servers — no local install required. We use the standard EBI Job Dispatcher REST endpoints:

| Step | URL pattern | Method |
|------|-------------|--------|
| Submit  | `https://www.ebi.ac.uk/Tools/services/rest/muscle/run/`           | `POST` |
| Status  | `https://www.ebi.ac.uk/Tools/services/rest/muscle/status/{jobid}` | `GET`  |
| Result  | `https://www.ebi.ac.uk/Tools/services/rest/muscle/result/{jobid}/aln-fasta` | `GET` |

Politeness rules (these matter — EBI will rate-limit and eventually block abusive clients):

- Always include a contact `email` in the submission. Put yours in the `EBI_EMAIL` variable above.
- Poll status no more than once every 5 seconds.
- Cache the result on disk and skip re-submission if the cached file already exists. We do this below.

In [ ]:
EBI_BASE = "https://www.ebi.ac.uk/Tools/services/rest/muscle"
ebi_aln    = os.path.join(WORK_DIR, "globin_ebi.aln")
ebi_jobid  = os.path.join(WORK_DIR, "globin_ebi_jobid.txt")

def ebi_submit(fasta_text, email):
    """POST a sequence to EBI MUSCLE; return the assigned job ID."""
    data = urlencode({
        "email":    email,
        "sequence": fasta_text,
        "format":   "fasta",
    }).encode()
    req = urllib.request.Request(f"{EBI_BASE}/run/", data=data, method="POST")
    with urllib.request.urlopen(req, timeout=60) as resp:
        return resp.read().decode().strip()

def ebi_status(job_id):
    """Return one of: RUNNING, FINISHED, FAILURE, ERROR, NOT_FOUND."""
    with urllib.request.urlopen(f"{EBI_BASE}/status/{job_id}", timeout=30) as resp:
        return resp.read().decode().strip()

def ebi_result(job_id, fmt="aln-fasta"):
    """Fetch the finished result in the requested format."""
    with urllib.request.urlopen(f"{EBI_BASE}/result/{job_id}/{fmt}", timeout=60) as resp:
        return resp.read().decode()

In [ ]:
# Run the EBI alignment, but cache aggressively: skip the network round-trip
# if we already have a result file on disk.
if os.path.exists(ebi_aln):
    print(f"Cache hit: {ebi_aln} already exists — skipping submission.")
else:
    if EBI_EMAIL.startswith("your.name@"):
        raise SystemExit("Please set EBI_EMAIL in cell 2 before submitting to EBI.")

    fasta_text = open(INPUT_FA).read()
    job_id = ebi_submit(fasta_text, EBI_EMAIL)
    print(f"Submitted EBI MUSCLE job: {job_id}")

    with open(ebi_jobid, "w") as g:
        g.write(job_id + "\n")

    # Poll, no faster than every 5 seconds.
    for attempt in range(60):
        status = ebi_status(job_id)
        print(f"  [{attempt:2d}] status = {status}")
        if status == "FINISHED":
            break
        if status in ("ERROR", "FAILURE", "NOT_FOUND"):
            raise SystemExit(f"EBI job {job_id} ended with status {status}")
        time.sleep(5)
    else:
        raise SystemExit("EBI job did not finish within 5 minutes")

    aln_text = ebi_result(job_id, fmt="aln-fasta")
    with open(ebi_aln, "w") as g:
        g.write(aln_text)
    print(f"Wrote {ebi_aln} ({os.path.getsize(ebi_aln)} bytes)")

In [ ]:
ebi_alignment = AlignIO.read(ebi_aln, "fasta")
print(f"EBI MUSCLE alignment:   {len(ebi_alignment)} sequences × {ebi_alignment.get_alignment_length()} columns")
print(f"Local MUSCLE alignment: {len(local_alignment)} sequences × {local_alignment.get_alignment_length()} columns")

# Quick column-by-column agreement on the *consensus residue per column*.
# This is a coarse measure: the two alignments need not have the same column
# count. We count how often the most common residue agrees in the leading
# min(L_local, L_ebi) columns.
def column_consensus(aln):
    L = aln.get_alignment_length()
    out = []
    for c in range(L):
        col = [str(r.seq[c]) for r in aln]
        most_common = Counter(col).most_common(1)[0][0]
        out.append(most_common)
    return out

cons_local = column_consensus(local_alignment)
cons_ebi   = column_consensus(ebi_alignment)
L_min = min(len(cons_local), len(cons_ebi))
agree = sum(1 for a, b in zip(cons_local[:L_min], cons_ebi[:L_min]) if a == b)
print(f"\nLeading-column consensus agreement: {agree}/{L_min} ({100*agree/L_min:.1f} %)")

**Question 1:** The two alignments come from the same input sequences but two different MUSCLE runs (a local binary vs. an EBI server). Are they identical? If not, where do they disagree, and what factors could explain the differences (algorithm version, default parameters, random tie-breaking in iterative refinement)?

*Answer hint:* MUSCLE's iterative refinement uses tree-based progressive alignment; small differences in the guide tree (which can be sensitive to floating-point arithmetic and library versions) propagate into different column choices in highly-gapped regions. Conserved core helices typically agree perfectly; loops disagree most.

## 4. Per-column conservation and gap fraction

Two complementary statistics tell you how well a column is conserved:

**Shannon entropy** of column $c$ (excluding gaps):

$$H_c = -\sum_{a \in \text{AA}} p_{a,c}\, \log_2 p_{a,c}$$

where $p_{a,c}$ is the frequency of amino acid $a$ in column $c$. A perfectly conserved column has $H_c = 0$; a uniform column has $H_c = \log_2 20 \approx 4.32$.

**Gap fraction** of column $c$:

$$g_c = \frac{\text{number of gaps in column } c}{N}$$

with $N$ the number of sequences. Columns where $g_c > 0.5$ are the kind of *insertions* HMMER would treat as Insert states (rather than Match states) when building a profile.

In [ ]:
# Use the local alignment as our canonical one going forward (the EBI alignment
# was just for comparison). Compute Shannon entropy and gap fraction per column.
canonical = local_alignment
L = canonical.get_alignment_length()
N = len(canonical)

entropy   = np.zeros(L)
gap_frac  = np.zeros(L)
for c in range(L):
    col   = [str(r.seq[c]) for r in canonical]
    n_gap = col.count("-")
    gap_frac[c] = n_gap / N
    aas = [a for a in col if a != "-"]
    if not aas:
        entropy[c] = 0.0
        continue
    counts = Counter(aas)
    p = np.array(list(counts.values())) / len(aas)
    entropy[c] = -np.sum(p * np.log2(p))

conservation = 1 - entropy / np.log2(20)   # 1 = fully conserved, 0 = uniform

# Helix overlay: we annotate the conservation plot with the canonical A–H
# helices of the globin fold. We use whichever reference sequence happens
# to be in the alignment. Sperm-whale myoglobin (P02185) is the gold
# standard reference; if it is not present (e.g. when the alignment came
# from 5_1's BLAST hit set, which is β-globin-only), fall back to human
# β-globin (P68871) — guaranteed to be in any β-globin-derived alignment.
HELIX_REFERENCES = [
    {
        "id_prefix":  "sp|P02185|MYG_PHYMC",
        "label":      "sperm-whale myoglobin",
        "ranges":     {"A": (3, 18),  "B": (20, 35),  "C": (37, 42), "D": (51, 57),
                       "E": (58, 77), "F": (86, 95),  "G": (100, 118), "H": (124, 149)},
        "distal_E7":  65,
        "proximal_F8": 94,
    },
    {
        "id_prefix":  "sp|P68871|HBB_HUMAN",
        "label":      "human β-globin",
        "ranges":     {"A": (6, 17),  "B": (21, 36),  "C": (37, 42), "D": (52, 57),
                       "E": (58, 76), "F": (84, 94),  "G": (99, 118),  "H": (124, 148)},
        "distal_E7":  64,
        "proximal_F8": 93,
    },
]

reference     = None
reference_seq = None
for ref in HELIX_REFERENCES:
    match = next((r for r in canonical if r.id.startswith(ref["id_prefix"])), None)
    if match is not None:
        reference, reference_seq = ref, match
        break

if reference is None:
    print("⚠️  No reference globin (P02185 or P68871) in the alignment — helix overlay disabled")
    helix_columns = {}
    aln2res       = []
else:
    print(f"Using {reference['label']} ({reference['id_prefix'].split('|')[1]}) as helix reference")
    pos = 0
    aln2res = []
    for ch in str(reference_seq.seq):
        if ch == "-":
            aln2res.append(None)
        else:
            pos += 1
            aln2res.append(pos)
    helix_columns = {h: [c for c, p in enumerate(aln2res) if p is not None and lo <= p <= hi]
                     for h, (lo, hi) in reference["ranges"].items()}

fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
ax[0].plot(conservation, lw=1.0, color="#2a7f8a")
ax[0].set_ylabel("Conservation\n(1 − H/log₂20)")
ax[0].set_ylim(0, 1)
ax[1].fill_between(np.arange(L), gap_frac, color="#b8602f", alpha=0.7)
ax[1].set_ylabel("Gap fraction")
ax[1].set_xlabel("Alignment column")
ax[1].set_ylim(0, 1)

# Overlay helix bars on the conservation panel
y_top = 1.05
for h, cols in helix_columns.items():
    if cols:
        ax[0].plot([min(cols), max(cols)], [y_top, y_top], lw=4, solid_capstyle="butt")
        ax[0].text((min(cols)+max(cols))/2, y_top + 0.04, h,
                   ha="center", va="bottom", fontsize=9)
ax[0].set_ylim(0, 1.18)
fig.suptitle("Per-column conservation and gap fraction (helices A–H overlaid)")
fig.tight_layout()
plt.show()

**Question 2:** Looking at the conservation curve and the helix overlay, which alignment regions correspond to the heme-binding helices E and F? Are they more conserved than the loops between helices? Where do you see the gap fraction spike, and what does that say about insertion/deletion tolerance in those regions?

*Answer hint:* The proximal histidine that coordinates heme iron sits inside helix F; the distal histidine sits in helix E. Both are functionally constrained, so the surrounding columns should show very low entropy (high conservation). The CD and EF loops, by contrast, are insert hotspots and have visibly more gaps.

## 5. Consensus and the heme-binding histidines

Globins fold around a heme group. Two histidines anchor the iron:

- **Distal His (E7)** — sits across the binding pocket, modulates O₂ affinity.
- **Proximal His (F8)** — coordinates the iron directly through Nε.

Their column positions in the alignment are the test of whether your MSA is biologically sensible: if those two positions are not 100% conserved, something is wrong with either your alignment or your sequence set.

In [ ]:
# Hand-rolled threshold-consensus: take the most common character (including
# '-') in each column; if it occurs in fewer than `threshold` of sequences,
# emit 'X' instead. This is what Biopython's now-removed AlignInfo
# .gap_consensus(threshold=...) used to compute, written out explicitly so
# we are not at the mercy of Biopython's API churn.
THRESHOLD = 0.5
consensus_chars = []
for c in range(canonical.get_alignment_length()):
    col = [str(r.seq[c]) for r in canonical]
    most_common, count = Counter(col).most_common(1)[0]
    consensus_chars.append(most_common if count / len(col) >= THRESHOLD else "X")
consensus = "".join(consensus_chars)

print(f"Consensus (threshold {THRESHOLD}):")
for i in range(0, len(consensus), 60):
    print(f"  {i+1:4d}  {consensus[i:i+60]}")

# Locate the two histidines in the consensus by mapping through the chosen
# reference globin (sperm-whale myoglobin or human β-globin, whichever is
# in the alignment). Both reference proteins share the globin fold and
# place the distal/proximal histidines at the same structural positions
# (E7 and F8), but at slightly different sequence positions because of
# different N-terminal lengths.
if reference is not None:
    distal_col   = next(c for c, p in enumerate(aln2res) if p == reference["distal_E7"])
    proximal_col = next(c for c, p in enumerate(aln2res) if p == reference["proximal_F8"])
    label = reference["label"]
    print(f"\nReference: {label}")
    print(f"\nDistal His (E7, {label} pos {reference['distal_E7']})   → column {distal_col}")
    print(f"  consensus character: {consensus[distal_col]}")
    print(f"  column residues:    {[str(r.seq[distal_col]) for r in canonical]}")
    print(f"\nProximal His (F8, {label} pos {reference['proximal_F8']}) → column {proximal_col}")
    print(f"  consensus character: {consensus[proximal_col]}")
    print(f"  column residues:    {[str(r.seq[proximal_col]) for r in canonical]}")

**Question 3:** Are the distal and proximal histidines 100% conserved across your alignment? If not, which sequence breaks the pattern, and what does that tell you about the biology of that organism's globin (or about a possible error in your sequence set)?

*Answer hint:* Some non-vertebrate globins (e.g. truncated globins, some neuroglobins) have substitutions at E7. The proximal His F8 is essentially universal — a non-His there usually means either you have a degenerate paralog, or your alignment placed the column wrong.

## 6. Save the canonical alignment for `5_3_phylogeny.ipynb`

The downstream phylogeny notebook reads the file we save here. We also persist a metadata table so the phylogeny notebook can label tips by species without re-parsing FASTA headers.

In [ ]:
canonical_path = os.path.join(WORK_DIR, "globin_msa.aln")
metadata_path  = os.path.join(WORK_DIR, "globin_msa_metadata.csv")

AlignIO.write(canonical, canonical_path, "fasta")
print(f"Wrote canonical alignment: {canonical_path} ({os.path.getsize(canonical_path)} bytes)")

# Build a metadata table from the SwissProt-style headers:
# >sp|ACC|ENTRY  Description OS=Species OX=TaxID GN=Gene PE=... SV=...
import re

rows = []
for r in canonical:
    m_acc = re.match(r"sp\|([A-Z0-9]+)\|(\S+)", r.id)
    desc  = r.description
    m_os  = re.search(r"OS=([^=]+?)\s+(?:OX=|GN=|PE=|SV=|$)", desc)
    rows.append({
        "alignment_id": r.id,
        "accession":    m_acc.group(1) if m_acc else "",
        "entry_name":   m_acc.group(2) if m_acc else "",
        "species":      m_os.group(1).strip() if m_os else "",
        "ungapped_len": len(str(r.seq).replace("-", "")),
    })

meta = pd.DataFrame(rows)
meta.to_csv(metadata_path, index=False)
print(f"Wrote metadata: {metadata_path}")
meta

**Question 4:** You aligned 12 sequences. If you added 50 more globins (more vertebrates, more invertebrates, more plants), would you expect the existing alignment columns to stay roughly fixed, or would the column count (and which residue lands in which column) change substantially? What implication does that have for a phylogeny built downstream?

*Answer hint:* Progressive MSA is **not** stable to added sequences — the guide tree changes when you add new leaves, and that can shift columns in flexible loop regions. A robust phylogeny rebuilds the alignment every time the sequence set changes; using a fixed reference profile (HMMER's approach in `6_3_hmmer_search.ipynb`) decouples alignment from set composition.

## Resources

| Topic | Reference |
|-------|-----------|
| MUSCLE algorithm | Edgar RC (2004). *Nucleic Acids Res* 32:1792. |
| MUSCLE 5 ensembles | Edgar RC (2022). *Nat Commun* 13:6968. |
| EBI Job Dispatcher | Madeira F et al. (2024). *Nucleic Acids Res* 52:W521. |
| EBI MUSCLE REST API | https://www.ebi.ac.uk/Tools/common/tools/help/index.html?context=muscle |
| MUSCLE GitHub | https://github.com/rcedgar/muscle |
| Course widget — pairwise DP | `docs/alignment-sandbox.html` |
| Course widget — substitution matrices | `docs/scoring-matrices.html` |
| Next notebook | `5_3_phylogeny.ipynb` (uses `msa_work/globin_msa.aln`) |